# DORA Document Extraction Demo

从 DORA (Digital Operational Resilience Act) 相关文档中提取结构化实体和事实。
使用 `extract_document()` Python API 直接调用,无需启动 HTTP server。

**Entity types**: Regulation, Organization, Requirement, ICTService, Risk

**Model**: Mistral Small (免费) 或 GPT-4.1

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

## 0. Setup

In [ ]:
import os, json
from pprint import pprint

# Pre-flight
for pkg in ('pydantic', 'diskcache', 'instructor', 'litellm'):
    try:
        __import__(pkg)
    except ImportError:
        raise RuntimeError(f'{pkg} not installed. Run: pip install -e ".[extraction]"')

api_key = os.environ.get('OPENAI_API_KEY') or os.environ.get('MISTRAL_API_KEY')
if not api_key:
    raise RuntimeError('Set OPENAI_API_KEY or MISTRAL_API_KEY')

from factpy_kernel.sdk import Entity, Field, Identity
from factpy_kernel.agent.extraction import extract_document, ExtractionDocumentError

print('Setup OK')

## 1. Define DORA Entity Schema

In [ ]:
class Regulation(Entity):
    title: str = Identity(primary_key=True)
    summary: str = Field(cardinality='single', description='What this regulation or article requires.')
    scope: str = Field(cardinality='single', description='Who or what this regulation applies to.')

class Organization(Entity):
    name: str = Identity(primary_key=True)
    role: str = Field(cardinality='single', description='Role: financial entity, ICT provider, regulator, or supervisory authority.')
    description: str = Field(cardinality='single', description='Brief description.')

class Requirement(Entity):
    name: str = Identity(primary_key=True)
    description: str = Field(cardinality='single', description='What must be done to comply.')
    category: str = Field(cardinality='single', description='Category: ICT risk management, incident reporting, resilience testing, third-party risk, or information sharing.')

class ICTService(Entity):
    name: str = Identity(primary_key=True)
    provider: str = Field(cardinality='single', description='Organization that provides this ICT service.')
    criticality: str = Field(cardinality='single', description='Criticality: critical, important, or standard.')

class Risk(Entity):
    name: str = Identity(primary_key=True)
    description: str = Field(cardinality='single', description='Description of this ICT-related risk.')
    mitigation: str = Field(cardinality='single', description='Required mitigation measure.')

SCHEMA_CLASSES = [Regulation, Organization, Requirement, ICTService, Risk]
ENTITY_DESCRIPTIONS = {
    'Regulation': 'A specific DORA article, chapter, or regulatory provision.',
    'Organization': 'A financial entity, ICT provider, regulator, or supervisory authority.',
    'Requirement': 'A compliance requirement imposed by DORA. NOT a general description.',
    'ICTService': 'A specific ICT service or system used by financial entities.',
    'Risk': 'An ICT-related operational risk identified in the document.',
}

print(f'Schema: {len(SCHEMA_CLASSES)} entity types')
for cls in SCHEMA_CLASSES:
    print(f'  {cls.__name__}')

## 2. Sample DORA Document

内置一段 DORA 相关的示例文本。你也可以替换为自己的文档。

In [ ]:
dora_text = """\
DORA - Digital Operational Resilience Act

Article 5 requires financial entities to establish an ICT risk management
framework. The framework must identify, classify, and mitigate ICT-related
risks on a continuous basis.

Article 17 mandates major ICT-related incident reporting to competent
authorities within 4 hours of classification. Financial entities must
submit initial, intermediate, and final reports.

Banks and insurance companies must conduct threat-led penetration testing
(TLPT) at least every 3 years, as specified in Article 26. Only qualified
external testers may perform TLPT on critical ICT systems.

ICT third-party service providers designated as critical by the ESAs are
subject to direct oversight. Cloud service providers like AWS, Azure, and
Google Cloud may be classified as critical ICT providers under Article 31.

The European Banking Authority (EBA), ESMA, and EIOPA jointly coordinate
the oversight framework for critical third-party providers.
"""

print(f'Document: {len(dora_text)} chars')

## 3. Extract

一个函数调用完成全链路: staging \u2192 LLM extraction \u2192 entity resolution

In [ ]:
try:
    result = extract_document(
        content=dora_text.encode('utf-8'),
        doc_name='dora_overview.txt',
        schema_classes=SCHEMA_CLASSES,
        model='mistral/mistral-small-latest',  # FREE; change to 'gpt-4.1' for OpenAI
        entity_descriptions=ENTITY_DESCRIPTIONS,
        enable_gleaning=True,
        enable_alias_merge=True,
    )
    print(f'\u2705 Extraction succeeded')
    print(f'  Model: {result.model}')
    print(f'  Segments: {result.staging_segments}')
    print(f'  Gleaning: {result.gleaning_segments_reexamined} segments re-examined')
    print(f'  Entities: {len(result.entities)}')
    print(f'  Facts: {len(result.facts)}')
    print(f'  Merges: {len(result.merge_events)}')
except ExtractionDocumentError as exc:
    print(f'\u274c Extraction failed at stage: {exc.stage}')
    print(f'  {exc}')

## 4. Entities

In [ ]:
print(f'Extracted {len(result.entities)} entities:\n')
for e in result.entities:
    etype = e['entity_type']
    identity = e['identity']
    name = list(identity.values())[0] if identity else '?'
    print(f'  {etype:15s}  {name:30s}  ({e["fact_count"]} facts)')

## 5. Facts

In [ ]:
print(f'Extracted {len(result.facts)} facts:\n')
for f in result.facts:
    etype = f.entity_type
    identity = list(f.entity_identity.values())[0] if f.entity_identity else '?'
    pred = f.pred_id.split(':')[1] if ':' in f.pred_id else f.pred_id
    values = [str(v) for _, v in f.field_values] if f.field_values else ['-']
    val_str = values[0][:70] if values else '-'
    print(f'  {etype:15s} {identity:25s} .{pred:15s} = {val_str}')

## 6. Metrics

In [ ]:
m = result.metrics
print(f'Segments:    {m.total_segments}')
print(f'Proposals:   {m.total_proposal_count}')
print(f'Valid:       {m.total_valid_count}')
print(f'Rejections:  {m.total_rejection_count}')
print(f'Duration:    {m.batch_duration_ms}ms')
print(f'Gleaning:    {result.gleaning_segments_reexamined} segments')
if result.merge_events:
    alias = sum(1 for e in result.merge_events if e.alias_merge)
    print(f'Merges:      {len(result.merge_events)} (alias: {alias})')

## 7. Try Your Own Document

替换 `dora_text` 为你自己的文本,或加载一个文件:

```python
with open('your_document.pdf', 'rb') as f:
    result = extract_document(
        content=f.read(),
        doc_name='your_document.pdf',
        schema_classes=SCHEMA_CLASSES,
        model='mistral/mistral-small-latest',
        entity_descriptions=ENTITY_DESCRIPTIONS,
    )
```